### Needs `01-ReferenceDFs.ipynb` to be executed first

Reads from:
- `gs_extractions_raw.json`
- `gs_extractions_raw_ynorm.json`
- `year_scope_df.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np

## Import Data

In [2]:
gs_extractions_raw  = pd.read_json("gs_extractions_raw.json")
gs_extractions_raw_ynorm = pd.read_json("gs_extractions_raw_ynorm.json")
year_scope_df       = pd.read_json("year_scope_df.json")

CATEGORIES = ["value", "unit"]
VARIANTS   = ["t1", "t2", "m1", "m2", "i1", "i2"]

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    gs_extractions_raw[col] = gs_extractions_raw[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    gs_extractions_raw_ynorm[col] = gs_extractions_raw_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [ ]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard




def eval_intern(source, result_basis) -> pd.DataFrame:
    result = result_basis.copy()
    
    for cat in CATEGORIES: # "value" then "unit" comparison
        for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
            result[f"{v}_{cat}_hit"] = source.apply(
                check_hit,                  # Apply check for every row
                args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    
    return result

def eval(source) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]])
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## Straight Forward

In [12]:
just_eval, expended = eval(gs_extractions_raw)

print_eval(just_eval)

### value ###
t1: True = 2078 || False = 130 || Quota = 94.11%
t2: True = 2087 || False = 121 || Quota = 94.52%
m1: True = 1607 || False = 601 || Quota = 72.78%
m2: True = 1923 || False = 285 || Quota = 87.09%
i1: True = 1874 || False = 334 || Quota = 84.87%
i2: True = 1858 || False = 350 || Quota = 84.15%

### unit ###
t1: True = 1952 || False = 256 || Quota = 88.41%
t2: True = 1935 || False = 273 || Quota = 87.64%
m1: True = 1764 || False = 444 || Quota = 79.89%
m2: True = 1787 || False = 421 || Quota = 80.93%
i1: True = 1761 || False = 447 || Quota = 79.76%
i2: True = 1744 || False = 464 || Quota = 78.99%



In [13]:
ynorm, expended_ynorm = eval(gs_extractions_raw_ynorm)

print_eval(ynorm)

### value ###
t1: True = 2084 || False = 124 || Quota = 94.38%
t2: True = 2100 || False = 108 || Quota = 95.11%
m1: True = 1587 || False = 621 || Quota = 71.88%
m2: True = 1920 || False = 288 || Quota = 86.96%
i1: True = 1884 || False = 324 || Quota = 85.33%
i2: True = 1870 || False = 338 || Quota = 84.69%

### unit ###
t1: True = 1965 || False = 243 || Quota = 88.99%
t2: True = 1956 || False = 252 || Quota = 88.59%
m1: True = 1755 || False = 453 || Quota = 79.48%
m2: True = 1786 || False = 422 || Quota = 80.89%
i1: True = 1777 || False = 431 || Quota = 80.48%
i2: True = 1760 || False = 448 || Quota = 79.71%



In [ ]:
# Just i1 data for GEPA
expended_ynorm[["report_name","status","year","scope","value","unit","unit_normalized","label","value_i1","unit_i1","i1_value_hit","i1_unit_hit"]].to_csv("just_i1.csv")

In [14]:
def save_eval_comparison(eval_raw, eval_ynorm, filepath="eval_comparison.json"):
    """Speichert beide Evaluationen strukturiert zum Vergleich"""
    results = []
    
    for cat in CATEGORIES:
        for v in VARIANTS:
            col_raw = eval_raw[f"{v}_{cat}_hit"]
            col_ynorm = eval_ynorm[f"{v}_{cat}_hit"]
            
            results.append({
                "variant": v,
                "category": cat,
                "raw_true": col_raw.value_counts()[True],
                "ynorm_true": col_ynorm.value_counts()[True],
                "raw_false": col_raw.value_counts()[False],
                "ynorm_false": col_ynorm.value_counts()[False],
                "abs_improvmnt": col_ynorm.value_counts()[True] - col_raw.value_counts()[True],
                "raw_quota": round(col_raw.mean() * 100, 2),
                "ynorm_quota": round(col_ynorm.mean() * 100, 2),
                "delta_quota": round((col_ynorm.mean() - col_raw.mean()) * 100, 2)
            })
    
    df_results = pd.DataFrame(results)
    df_results.to_json(filepath, orient="records", indent=2)
    df_results.to_csv(filepath.replace(".json", ".csv"), index=False)
    
    return df_results

# Speichern und anzeigen
comparison = save_eval_comparison(just_eval, ynorm)
print(comparison)


   variant category  raw_true  ynorm_true  raw_false  ynorm_false  \
0       t1    value      2078        2084        130          124   
1       t2    value      2087        2100        121          108   
2       m1    value      1607        1587        601          621   
3       m2    value      1923        1920        285          288   
4       i1    value      1874        1884        334          324   
5       i2    value      1858        1870        350          338   
6       t1     unit      1952        1965        256          243   
7       t2     unit      1935        1956        273          252   
8       m1     unit      1764        1755        444          453   
9       m2     unit      1787        1786        421          422   
10      i1     unit      1761        1777        447          431   
11      i2     unit      1744        1760        464          448   

    abs_improvmnt  raw_quota  ynorm_quota  delta_quota  
0               6      94.11        94.38    

# Detailed Degradation Analysis (Claude!)

In [ ]:
def analyze_degradation_detailed(raw_source, ynorm_source, raw_eval, ynorm_eval, variant, category):
    """Detaillierte Analyse: zeigt Gold-Standard vs. was raw/ynorm extrahiert haben"""

    hit_col = f"{variant}_{category}_hit"
    value_col = f"value_{variant}"
    unit_col = f"unit_{variant}"
    gs_col = category  # "value" or "unit"

    # Rows wo raw=True aber ynorm=False (Verschlechterung)
    degraded_mask = (raw_eval[hit_col] == True) & (ynorm_eval[hit_col] == False)
    degraded_idx = raw_source[degraded_mask].index.tolist()

    results = []
    for idx in degraded_idx:
        gs_val = raw_source.iloc[idx][gs_col]
        raw_val = raw_source.iloc[idx][value_col]
        ynorm_val = ynorm_source.iloc[idx][value_col]
        
        results.append({
            "report_name": raw_source.iloc[idx]["report_name"],
            "year": raw_source.iloc[idx]["year"],
            "scope": raw_source.iloc[idx]["scope"],
            "gs_value": gs_val,
            "raw_extracted": raw_val,
            "ynorm_extracted": ynorm_val,
            "raw_was_list": isinstance(raw_val, list),
            "ynorm_was_list": isinstance(ynorm_val, list),
        })

    return pd.DataFrame(results)


# m1 value: detaillierte Analyse
print("=" * 110)
print("M1 VALUE DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m1_value_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m1", "value")
print(f"Total degraded: {len(m1_value_details)}\n")
if len(m1_value_details) > 0:
    print(m1_value_details.to_string())
    m1_value_details.to_csv("m1_value_degradations.csv", index=False)
    print("\n✓ Saved to: m1_value_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M2 VALUE DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m2_value_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m2", "value")
print(f"Total degraded: {len(m2_value_details)}\n")
if len(m2_value_details) > 0:
    print(m2_value_details.to_string())
    m2_value_details.to_csv("m2_value_degradations.csv", index=False)
    print("\n✓ Saved to: m2_value_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M1 UNIT DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m1_unit_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m1", "unit")
print(f"Total degraded: {len(m1_unit_details)}\n")
if len(m1_unit_details) > 0:
    print(m1_unit_details.to_string())
    m1_unit_details.to_csv("m1_unit_degradations.csv", index=False)
    print("\n✓ Saved to: m1_unit_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M2 UNIT DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m2_unit_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m2", "unit")
print(f"Total degraded: {len(m2_unit_details)}\n")
if len(m2_unit_details) > 0:
    print(m2_unit_details.to_string())
    m2_unit_details.to_csv("m2_unit_degradations.csv", index=False)
    print("\n✓ Saved to: m2_unit_degradations.csv")
else:
    print("(No degradations)")

m1 and m2 had indeed extracted values that the gold standard did not aknowledge. Therefore a miss-mapping of the "year"-value, which resolved to "NaN" led to correct retrieval. But the fact that it had found a value, which is now shown, the score went down.

In [ ]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list):
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col])

In [ ]:
# To give GEPA the reason in what ways the extraction did not match the Gold Standard
def check_hit_with_detail(row, extraction_col, gs_col) -> tuple[bool, dict | None]:
    if isinstance(row[extraction_col], list):
        hit = row[gs_col] in row[extraction_col]
        if not hit:
            return False, {
                "expected": row[gs_col],
                "got": row[extraction_col],
                "report": row.get("report_name"),
                "scope": row.get("scope"),
            }
        return True, None # The mandated value by GS was found by the extraction 
    else:
        hit = pd.isna(row[gs_col])
        if not hit:
            return False, {
                "expected": row[gs_col],
                "got": None,  # Extraktion hat nichts gefunden
                "report": row.get("report_name"),
                "scope": row.get("scope"),
            }
        return True, None # Nothing mandated by GS, nothing found by the extraction


Within GEPA's "evaluate"

In [ ]:
VARIANT_OA = "i1"  # nur für GEPA-Optimierung

def evaluate(candidate: str) -> tuple[float, dict]:
    results = run_pipeline_with(candidate)  # nur i1 ausführen

    hits = 0
    total = 0
    misses = {cat: [] for cat in CATEGORIES}

    for _, row in results.iterrows():
        for cat in CATEGORIES:
            total += 1
            hit, detail = check_hit_with_detail(row, f"{cat}_i1", cat)
            if hit:
                hits += 1
            elif detail:
                misses[cat].append(detail)

    misses_clean = {cat: errs for cat, errs in misses.items() if errs}
    return hits / total, {"misses": misses_clean}